# 05 - XLS to Parquet

Este notebook reconstruye los archivos **`.xls` / `.xlsx`** de producción de leche y los guarda como un único archivo **Parquet** en `data/interim/`.

## Qué hace
- localiza archivos Excel en `data/raw/xls/`
- lee hojas con **encabezado multinivel** (`header=[0, 1]`)
- aplana y normaliza nombres de columnas
- convierte columnas de fecha/hora y tiempos
- elimina filas/columnas completamente vacías
- concatena todos los archivos
- guarda `milking.parquet`

## Qué no hace
- no hace feature engineering
- no hace merges con clima, rumia o PDFs
- no imputa valores faltantes


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
import unicodedata

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)

## 1. Resolver rutas del proyecto

In [2]:
XLS_DIR = Path("../data/raw/produccion")
INTERIM_DIR = Path("../data/interim")

print("XLS_DIR:", XLS_DIR)
print("INTERIM_DIR:", INTERIM_DIR)

XLS_DIR: ../data/raw/produccion
INTERIM_DIR: ../data/interim


## 2. Localizar archivos Excel

In [3]:
xls_files = list(XLS_DIR.glob("*.xls")) + list(XLS_DIR.glob("*.xlsx"))

xls_files = [
    f for f in xls_files
    if f.name.lower().startswith("producciones de leche")
]

print("Archivos válidos:", len(xls_files))
for f in xls_files:
    print(f.name)

Archivos válidos: 65
Producciones de leche 6137.xls
Producciones de leche 8708.xls
Producciones de leche 1510.xls
Producciones de leche 6134.xls
Producciones de leche 2150.xls
Producciones de leche 6054.xls
Producciones de leche 6050.xls
Producciones de leche 5767.xls
Producciones de leche 6124.xls
Producciones de leche 6131.xls
Producciones de leche1204.xls
Producciones de leche 6092.xls
Producciones de leche 8732.xls
Producciones de leche 5981.xls
Producciones de leche 1216.xls
Producciones de leche 2119.xls
Producciones de leche 6197.xls
Producciones de leche 6154.xls
Producciones de leche 2087.xls
Producciones de leche 8755.xls
Producciones de leche 1567.xls
Producciones de leche 6194.xls
Producciones de leche 1638.xls
Producciones de leche 1228.xls
Producciones de leche 2122.xls
Producciones de leche 1577.xls
Producciones de leche 6226.xls
Producciones de leche 1213.xls
Producciones de leche 2108.xls
Producciones de leche 6178.xls
Producciones de leche 1617.xls
Producciones de lec

## 3. Funciones auxiliares

In [4]:
def StripAccents(text: str) -> str:
    text = unicodedata.normalize("NFKD", text)
    return "".join(ch for ch in text if not unicodedata.combining(ch))


def CleanColumnName(name: str) -> str:
    name = str(name).strip()
    name = name.replace("\n", " ")
    name = StripAccents(name)
    name = name.lower()

    # reemplazos específicos
    replacements = {
        "hora de inicio": "hora_inicio",
        "numero de ordeno": "numero_ordeno",
        "duracion (mm:ss)": "duracion_mmss",
        "produccion (kg)": "produccion_kg",
        "intervalo de ordeno (hh:mm)": "intervalo_ordeno_hhmm",
        "rcs (* 1000 celulas / ml)": "rcs",
        "destino leche": "destino_leche",
        "razon de la desviacion": "razon_desviacion",
        "programa de lavado": "programa_lavado",
        "pezon": "pezon",
        "usuario": "usuario",
        "ubre": "ubre",
        "accion": "accion",
        "patada": "patada",
        "incompleto": "incompleto",
        "pezones no encontrados": "pezones_no_encontrados",
        "ms": "ms",
        "di": "di",
        "dd": "dd",
        "ti": "ti",
        "td": "td",
        "eo/po": "eo_po",
    }
    name = replacements.get(name, name)

    name = re.sub(r"\s+", "_", name)
    name = re.sub(r"[^a-z0-9_]", "", name)
    name = re.sub(r"_+", "_", name).strip("_")
    return name


def FlattenColumns(columns) -> pd.Index:
    flat = []

    for top, bottom in columns:
        top = "" if pd.isna(top) else str(top).strip()
        bottom = "" if pd.isna(bottom) else str(bottom).strip()

        if top.lower().startswith("unnamed"):
            top = ""
        if bottom.lower().startswith("unnamed"):
            bottom = ""

        # En este dataset casi siempre nos conviene el nombre "bottom"
        # porque la fila inferior contiene la variable real.
        if bottom:
            name = bottom
        else:
            name = top

        flat.append(CleanColumnName(name))

    return pd.Index(flat)


def NormalizeSpanishAmPm(series: pd.Series) -> pd.Series:
    s = series.astype(str)
    s = (
        s.str.replace("\n", " ", regex=False)
         .str.replace(r"\s+", " ", regex=True)
         .str.replace(r"a\.?\s*m\.?", "AM", regex=True)
         .str.replace(r"p\.?\s*m\.?", "PM", regex=True)
         .str.strip()
    )
    return s


def ParseDatetimeColumn(series: pd.Series) -> pd.Series:
    s = NormalizeSpanishAmPm(series)
    return pd.to_datetime(s, dayfirst=True, errors="coerce")


def ParseTimedeltaMinutes(series: pd.Series) -> pd.Series:
    s = series.astype(str).str.strip()
    td = pd.to_timedelta(s, errors="coerce")
    return td.dt.total_seconds() / 60.0

def ExtractCowID(filename):

    m = re.search(r"(\d+)", filename)

    if m:
        return int(m.group(1))

    return None

def LoadExcel(file_path: Path) -> pd.DataFrame:
    xls = pd.ExcelFile(file_path)
    dfs = []

    for sheet in xls.sheet_names:
        try:
            df = pd.read_excel(file_path, sheet_name=sheet, header=[0, 1])
            df.columns = FlattenColumns(df.columns)

            df["cow_id"] = ExtractCowID(file_path.name)
            df["source_file"] = file_path.name

            # limpieza estructural mínima
            df = df.dropna(axis=1, how="all")
            df = df.dropna(axis=0, how="all")

            # convertir datetime si existe
            if "hora_inicio" in df.columns:
                df["hora_inicio"] = ParseDatetimeColumn(df["hora_inicio"])

            # filtrar filas válidas si existe hora_inicio
            if "hora_inicio" in df.columns:
                df = df[df["hora_inicio"].notna()]

            # convertir duraciones / intervalos a minutos
            if "duracion_mmss" in df.columns:
                df["duracion_min"] = ParseTimedeltaMinutes(df["duracion_mmss"])

            if "intervalo_ordeno_hhmm" in df.columns:
                df["intervalo_ordeno_min"] = ParseTimedeltaMinutes(df["intervalo_ordeno_hhmm"])

            dfs.append(df)

        except Exception as e:
            print(f"Error en archivo={file_path.name} hoja={sheet}: {e}")

    if not dfs:
        return pd.DataFrame()

    return pd.concat(dfs, ignore_index=True)

## 4. Probar con un archivo de ejemplo

In [5]:
if not xls_files:
    raise RuntimeError("No se encontraron archivos Excel en data/raw/xls")

sample_df = LoadExcel(xls_files[0])

print("Archivo ejemplo:", xls_files[0].name)
print("Shape:", sample_df.shape)
sample_df.head()

Archivo ejemplo: Producciones de leche 6137.xls
Shape: (427, 17)


/var/folders/8n/ss0jgxw12y974534qchnd3jw0000gn/T/ipykernel_87160/2830172264.py:83: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  return pd.to_datetime(s, dayfirst=True, errors="coerce")


,hora_inicio,numero_ordeno,duracion_mmss,produccion_kg,intervalo_ordeno_hhmm,di,dd,td,ubre,pezon,destino_leche,programa_lavado,ms,cow_id,source_file,duracion_min,intervalo_ordeno_min
0,2025-01-01 00:54:00,1,06:22,16.72,10:56,4.44,5.57,6.71,0,NaN,Tanque,NaN,VMS 1,6137,Producciones de leche 6137.xls,NaN,NaN
1,2025-01-01 09:46:00,2,04:58,14.25,08:46,4.13,4.85,5.27,0,NaN,Tanque,NaN,VMS 1,6137,Producciones de leche 6137.xls,NaN,NaN
2,2025-01-01 21:05:00,3,05:53,16.48,11:13,5.76,4.11,6.61,0,NaN,Tanque,NaN,VMS 1,6137,Producciones de leche 6137.xls,NaN,NaN
3,2025-01-02 08:46:00,1,08:14,18.52,11:34,5.67,5.76,7.09,0,NaN,Tanque,NaN,VMS 1,6137,Producciones de leche 6137.xls,NaN,NaN
4,2025-01-02 15:09:00,2,04:06,9.68,06:14,3.12,3.03,3.53,0,NaN,Tanque,NaN,VMS 1,6137,Producciones de leche 6137.xls,NaN,NaN


In [6]:
print(sample_df.columns.tolist())

['hora_inicio', 'numero_ordeno', 'duracion_mmss', 'produccion_kg', 'intervalo_ordeno_hhmm', 'di', 'dd', 'td', 'ubre', 'pezon', 'destino_leche', 'programa_lavado', 'ms', 'cow_id', 'source_file', 'duracion_min', 'intervalo_ordeno_min']


## 5. Cargar y concatenar todos los Excel

In [7]:
dfs = []

for file_path in xls_files:
    df_file = LoadExcel(file_path)
    if not df_file.empty:
        dfs.append(df_file)
        print(f"OK  {file_path.name:40} -> {df_file.shape}")
    else:
        print(f"VACIO {file_path.name}")

if not dfs:
    raise RuntimeError("No se pudo cargar ningún archivo Excel válido.")



df = pd.concat(dfs, ignore_index=True)

print("\nDataset combinado:", df.shape)

OK  Producciones de leche 6137.xls           -> (427, 17)
OK  Producciones de leche 8708.xls           -> (479, 19)
OK  Producciones de leche 1510.xls           -> (42, 18)


/var/folders/8n/ss0jgxw12y974534qchnd3jw0000gn/T/ipykernel_87160/2830172264.py:83: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  return pd.to_datetime(s, dayfirst=True, errors="coerce")
/var/folders/8n/ss0jgxw12y974534qchnd3jw0000gn/T/ipykernel_87160/2830172264.py:83: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  return pd.to_datetime(s, dayfirst=True, errors="coerce")
/var/folders/8n/ss0jgxw12y974534qchnd3jw0000gn/T/ipykernel_87160/2830172264.py:83: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  return pd.to_datetime(s, dayfirst=True, errors="coerce")
/var/folders/8n/ss0jgx

OK  Producciones de leche 6134.xls           -> (433, 17)
OK  Producciones de leche 2150.xls           -> (357, 19)
OK  Producciones de leche 6054.xls           -> (470, 17)
OK  Producciones de leche 6050.xls           -> (360, 17)
OK  Producciones de leche 5767.xls           -> (382, 17)
OK  Producciones de leche 6124.xls           -> (219, 17)
OK  Producciones de leche 6131.xls           -> (425, 17)
OK  Producciones de leche1204.xls            -> (301, 19)
OK  Producciones de leche 6092.xls           -> (457, 17)
OK  Producciones de leche 8732.xls           -> (366, 18)
OK  Producciones de leche 5981.xls           -> (13, 17)
OK  Producciones de leche 1216.xls           -> (390, 17)
OK  Producciones de leche 2119.xls           -> (377, 18)
OK  Producciones de leche 6197.xls           -> (363, 17)
OK  Producciones de leche 6154.xls           -> (162, 18)


/var/folders/8n/ss0jgxw12y974534qchnd3jw0000gn/T/ipykernel_87160/2830172264.py:83: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  return pd.to_datetime(s, dayfirst=True, errors="coerce")
/var/folders/8n/ss0jgxw12y974534qchnd3jw0000gn/T/ipykernel_87160/2830172264.py:83: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  return pd.to_datetime(s, dayfirst=True, errors="coerce")
/var/folders/8n/ss0jgxw12y974534qchnd3jw0000gn/T/ipykernel_87160/2830172264.py:83: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  return pd.to_datetime(s, dayfirst=True, errors="coerce")
/var/folders/8n/ss0jgx

OK  Producciones de leche 2087.xls           -> (475, 17)
OK  Producciones de leche 8755.xls           -> (320, 19)
OK  Producciones de leche 1567.xls           -> (450, 18)
OK  Producciones de leche 6194.xls           -> (268, 18)
OK  Producciones de leche 1638.xls           -> (374, 17)
OK  Producciones de leche 1228.xls           -> (528, 17)
OK  Producciones de leche 2122.xls           -> (383, 18)
OK  Producciones de leche 1577.xls           -> (454, 18)
OK  Producciones de leche 6226.xls           -> (530, 17)
OK  Producciones de leche 1213.xls           -> (270, 17)
OK  Producciones de leche 2108.xls           -> (353, 17)
OK  Producciones de leche 6178.xls           -> (309, 18)
OK  Producciones de leche 1617.xls           -> (186, 17)
OK  Producciones de leche 6193.xls           -> (410, 18)
OK  Producciones de leche 2082.xls           -> (338, 17)
OK  Producciones de leche 1560.xls           -> (485, 17)
OK  Producciones de leche 2070.xls           -> (448, 19)
OK  Produccion

/var/folders/8n/ss0jgxw12y974534qchnd3jw0000gn/T/ipykernel_87160/2830172264.py:83: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  return pd.to_datetime(s, dayfirst=True, errors="coerce")
/var/folders/8n/ss0jgxw12y974534qchnd3jw0000gn/T/ipykernel_87160/2830172264.py:83: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  return pd.to_datetime(s, dayfirst=True, errors="coerce")


OK  Producciones de leche 2076.xls           -> (410, 17)
OK  Producciones de leche 1225.xls           -> (437, 17)
OK  Producciones de leche 6164.xls           -> (584, 16)
OK  Producciones de leche 1555.xls           -> (475, 19)
OK  Producciones de leche 2074.xls           -> (404, 16)
OK  Producciones de leche 8714.xls           -> (414, 17)
OK  Producciones de leche 1644.xls           -> (366, 19)
OK  Producciones de leche 6062.xls           -> (342, 18)
OK  Producciones de leche 8715.xls           -> (357, 18)
OK  Producciones de leche 8703.xls           -> (493, 17)
OK  Producciones de leche 1243.xls           -> (428, 17)
OK  Producciones de leche 2165.xls           -> (284, 17)
OK  Producciones de leche 1242.xls           -> (439, 17)
OK  Producciones de leche 8712.xls           -> (380, 18)
OK  Producciones de leche 1497.xls           -> (12, 16)
OK  Producciones de leche 8707.xls           -> (408, 17)
OK  Producciones de leche 6070.xls           -> (411, 18)
OK  Produccione

/var/folders/8n/ss0jgxw12y974534qchnd3jw0000gn/T/ipykernel_87160/2830172264.py:83: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  return pd.to_datetime(s, dayfirst=True, errors="coerce")
/var/folders/8n/ss0jgxw12y974534qchnd3jw0000gn/T/ipykernel_87160/2830172264.py:83: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  return pd.to_datetime(s, dayfirst=True, errors="coerce")
/var/folders/8n/ss0jgxw12y974534qchnd3jw0000gn/T/ipykernel_87160/2830172264.py:83: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  return pd.to_datetime(s, dayfirst=True, errors="coerce")


## 6. Revisión rápida

In [8]:
print(df.shape)
print(df.columns.tolist())

(23763, 19)
['hora_inicio', 'numero_ordeno', 'duracion_mmss', 'produccion_kg', 'intervalo_ordeno_hhmm', 'di', 'dd', 'td', 'ubre', 'pezon', 'destino_leche', 'programa_lavado', 'ms', 'cow_id', 'source_file', 'duracion_min', 'intervalo_ordeno_min', 'ti', 'razon_desviacion']


In [9]:
df.columns.tolist()

['hora_inicio',
 'numero_ordeno',
 'duracion_mmss',
 'produccion_kg',
 'intervalo_ordeno_hhmm',
 'di',
 'dd',
 'td',
 'ubre',
 'pezon',
 'destino_leche',
 'programa_lavado',
 'ms',
 'cow_id',
 'source_file',
 'duracion_min',
 'intervalo_ordeno_min',
 'ti',
 'razon_desviacion']

In [10]:
nan_report = pd.DataFrame({
    "column": df.columns,
    "nan_count": df.isna().sum().values,
    "nan_pct": (df.isna().mean() * 100).values
}).sort_values("nan_pct", ascending=False)

nan_report.head(30)

,column,nan_count,nan_pct
16,intervalo_ordeno_min,23763,100.000000
15,duracion_min,23763,100.000000
18,razon_desviacion,23742,99.911627
11,programa_lavado,23074,97.100534
9,pezon,21458,90.300046
17,ti,1804,7.591634
6,dd,620,2.609098
5,di,33,0.138871
7,td,13,0.054707
8,ubre,0,0.000000


In [11]:
cols_to_drop_now = ["razon_desviacion", "programa_lavado"]

df = df.drop(columns=[c for c in cols_to_drop_now if c in df.columns])

In [12]:
df["pezon"].value_counts(dropna=False).head(20)

pezon
NaN         21458
TD            812
DI            770
TI            271
DD            209
DD,TD          55
DI,TD          54
DI,DD          32
DI,DD,TI       27
TI,TD          25
DI,TI          23
DD,TI          17
DD,TI,TD        4
DI,TI,TD        3
DI,DD,TD        2
Todos           1
Name: count, dtype: int64

In [13]:
df["date"] = (
    pd.to_datetime(df["hora_inicio"])
    .dt.date
)

df.drop(columns=["hora_inicio"], inplace=True)



In [14]:
df.head()

,numero_ordeno,duracion_mmss,produccion_kg,intervalo_ordeno_hhmm,di,dd,td,ubre,pezon,destino_leche,ms,cow_id,source_file,duracion_min,intervalo_ordeno_min,ti,date
0,1,06:22,16.72,10:56,4.44,5.57,6.71,0,NaN,Tanque,VMS 1,6137,Producciones de leche 6137.xls,NaN,NaN,NaN,2025-01-01
1,2,04:58,14.25,08:46,4.13,4.85,5.27,0,NaN,Tanque,VMS 1,6137,Producciones de leche 6137.xls,NaN,NaN,NaN,2025-01-01
2,3,05:53,16.48,11:13,5.76,4.11,6.61,0,NaN,Tanque,VMS 1,6137,Producciones de leche 6137.xls,NaN,NaN,NaN,2025-01-01
3,1,08:14,18.52,11:34,5.67,5.76,7.09,0,NaN,Tanque,VMS 1,6137,Producciones de leche 6137.xls,NaN,NaN,NaN,2025-01-02
4,2,04:06,9.68,06:14,3.12,3.03,3.53,0,NaN,Tanque,VMS 1,6137,Producciones de leche 6137.xls,NaN,NaN,NaN,2025-01-02


## 8. Guardar Parquet

In [15]:
INTERIM_DIR.mkdir(parents=True, exist_ok=True)
output_path = INTERIM_DIR / "milking.parquet"

df.to_parquet(output_path, index=False)

print("Archivo guardado en:", output_path)
print("Filas:", len(df))
print("Columnas:", len(df.columns))

Archivo guardado en: ../data/interim/milking.parquet
Filas: 23763
Columnas: 17


## 9. Próximo paso sugerido

Después de este notebook, el siguiente paso lógico es unir:

- `milking.parquet`
- `rumination.parquet`
- `weather.parquet`
- `pdf_events.parquet`

en un notebook como `06_merge_sources.ipynb`.
